In [ ]:
# Cell 0 — Install dependencies (Unsloth canonical Colab branch) + clone feat/corpo
#
# Mirrors the install logic from Unsloth's Qwen2.5_(3B)-GRPO notebook, cells 4+5:
#   https://raw.githubusercontent.com/unslothai/notebooks/main/nb/Qwen2.5_(3B)-GRPO.ipynb
#
# Why this exact shape (do not "simplify"):
#   - UNSLOTH_VLLM_STANDBY=1   → +30% context-length headroom (unsloth-specific)
#   - upgrade `uv` first       → uv's resolver is stricter than pip's; avoids the
#                                pip-picks-wrong-trl mistakes we hit on Path B
#   - GPU-aware vllm/triton    → T4 needs vllm==0.9.2 + triton==3.2.0; A100/L4/H100
#                                use vllm==0.15.1 + latest triton. Latest vllm
#                                doesn't work on T4 (silent crash on import).
#   - ONE uv-call bundle       → vllm + numpy + pil + torchvision + bitsandbytes +
#                                xformers + unsloth resolved together so versions
#                                stay consistent (no pip-installs-X-then-uv-overrides-Y).
#   - trl==0.22.2 --no-deps    → avoids TRL 0.24's vllm_ascend + mergekit imports.
#                                --no-deps so trl doesn't drag in transitive packages
#                                that would fight the unsloth-bundled versions.
#   - transformers==4.56.2     → matches what unsloth's wheel was built against.
#   - peft==0.17.1 --no-deps   → 0.18+ added _maybe_shard_state_dict_for_tp which
#                                imports transformers.integrations.tensor_parallel.
#                                EmbeddingParallel — that symbol doesn't exist
#                                until transformers 4.57+. Pinning to 0.17.1 (last
#                                pre-TP release, 2025-08-21) avoids the conflict.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    # Non-Colab environment (local dev, RunPod, etc.): unsloth's simple path
    !pip install unsloth vllm
else:
    # Resolve currently-installed numpy + pillow versions to avoid churning them
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil   = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"

    # GPU-aware vllm + triton pinning
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except Exception:
        is_t4 = False
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")

    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"

!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq --no-deps peft==0.17.1

# Project-specific extras (not part of the canonical Unsloth recipe):
#   openai — DeepSeek V4-Pro judge HTTP client
!uv pip install -qqq "openai>=1.0.0"

# Clone the project at feat/corpo (private repo)
from google.colab import userdata
GITHUB_PAT = userdata.get('GITHUB_PAT')
!rm -rf /content/sft
!git clone --branch feat/corpo --depth 1 \
    https://{GITHUB_PAT}@github.com/deepanathanrajendiran-hub/sft-code-review.git \
    /content/sft
!cp /content/sft/*.py /content/sft/pyproject.toml /content/
!cp -r /content/sft/tests /content/
os.chdir("/content")

!ls /content/corpo_*.py /content/swecare_*.py /content/ood_metrics.py
print("\nVersion check:")
import trl, vllm, torch, datasets, peft, transformers
print(f"  trl          : {trl.__version__}     (expected 0.22.2)")
print(f"  transformers : {transformers.__version__}    (expected 4.56.2)")
print(f"  vllm         : {vllm.__version__}     (T4=0.9.2, else=0.15.1)")
print(f"  torch        : {torch.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  peft         : {peft.__version__}    (expected 0.17.1)")

In [ ]:
# Cell 1 — Mount Drive, load secrets, verify v4 backup
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

# v4 adapter paths — USER MUST ensure backup exists before this cell runs
V4_ADAPTER = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces'
V4_BACKUP  = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup'
assert os.path.exists(V4_BACKUP + '/adapter_config.json'), \
    f"v4 backup missing! Create one BEFORE running: cp -r {V4_ADAPTER} {V4_BACKUP}"
print(f"v4 adapter:  {V4_ADAPTER}")
print(f"v4 backup:   {V4_BACKUP}")

In [ ]:
# Cell 2 — Build train/eval splits, extract CLEAN defect labels, and score v4 (THE GATE)
# v5: no base-sample cache / no opponent. We extract clean, grounded defect tuples from the
# human PR threads (label_defects.py) and measure v4's JUDGE-INDEPENDENT recall + hallucination.
# Requires DEEPSEEK_API_KEY (Cell 1). No GPU needed for this cell.
import json, os, random

# --- training prompts from dev split; eval from test split (disjoint) ---
!python /content/swecare_loader.py --split dev \
    --output /content/ood_dev_prompts_raw.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl
with open('/content/ood_dev_prompts_raw.jsonl') as f:
    dev_rows = [json.loads(l) for l in f if l.strip()]
sample = random.Random(42).sample(dev_rows, min(1500, len(dev_rows)))
with open('/content/ood_train_prompts.jsonl', 'w') as f:
    for r in sample: f.write(json.dumps(r) + '\n')
!python /content/swecare_loader.py --split test \
    --output /content/ood_input.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl
print(f"dev pool {len(dev_rows)}   train sample {len(sample)}")

# --- Stage 1: extract clean defect tuples (drops questions/replies/style nits; grounds to diff) ---
os.makedirs('/content/cache', exist_ok=True)
!python /content/label_defects.py --input /content/ood_train_prompts.jsonl --output /content/cache/defect_labels_train.jsonl
!python /content/label_defects.py --input /content/ood_input.jsonl        --output /content/cache/defect_labels_eval.jsonl

# --- THE GATE: v4 (and base) recall + hallucination on the clean EVAL labels ---
# Uses the existing ood_preds_v4.jsonl (already has v4_pred + base_pred).
!python /content/score_v5.py \
    --preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/cache/defect_labels_eval.jsonl \
    --pred-fields v4_pred base_pred

print("\n[GATE] Decision:")
print("  - v4 recall well below 1.0  -> headroom exists, RL is viable. Record v4 recall + halluc; proceed to Cell 3.")
print("  - v4 recall already ~saturated -> RL can only restore, not exceed. STOP and pivot to data (v4.1).")
print("  - This v4 recall/halluc pair IS the bar Cell 5/Cell 7 must beat (recall UP, halluc <= v4).")

In [ ]:
# Cell 3 — Pre-training variance gate + auto-pick R_min for v5 (~10-15 min, ~$1 DeepSeek)
#
# v5 scores v4 rollouts with the VERIFIABLE reward (F1 on labeled + claim-penalty on clean +
# grounding + length) — no opponent, no quality judge. The gate confirms the reward has
# within-group spread (>=0.10) so advantages don't collapse, and prints p25/p33/p40/p50
# R_min candidates (CoRPO correctness boundary). This cell auto-extracts the p33 default.
import subprocess, re, sys

print("[cell3] running v5 variance gate (verifiable reward against clean defect labels)...")
result = subprocess.run(
    ["python", "/content/corpo_train.py", "--variance-gate-only",
     "--v4-adapter", V4_ADAPTER,
     "--v4-backup",  V4_BACKUP,
     "--train-prompts", "/content/ood_train_prompts.jsonl",
     "--defect-labels", "/content/cache/defect_labels_train.jsonl",
     "--output-dir", "/content/corpo-out"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr, file=sys.stderr)

# Distinguish a real gate FAIL (flat reward) from a crash (code/env error). The gate only
# prints "verdict: PASS/FAIL" if it ran to completion; a traceback means it crashed.
reached_verdict = ("[variance-gate] verdict:" in result.stderr)
if result.returncode != 0:
    if not reached_verdict:
        raise RuntimeError(
            "[variance-gate] CRASHED before a verdict (see traceback above) — a code/env error, "
            "NOT a flat-reward failure. Fix the error (e.g. git pull + re-copy /content/*.py) and re-run."
        )
    raise RuntimeError(
        "[variance-gate] FAIL — within-group reward std < 0.10 (flat reward). Do NOT train. "
        "Inspect the histogram above; the fix is to rebalance the reward, not to train. Paste the output."
    )

m = re.search(r"recommended default: p33 = ([\d.]+)", result.stderr)
if not m:
    raise RuntimeError("[variance-gate] PASSED but couldn't parse R_min; set R_MIN manually from the printout.")
R_MIN = float(m.group(1))
print(f"\n[cell3] PASS — auto-selected R_MIN = {R_MIN}  (p33 of the v4 verifiable-reward distribution)")
print(f"[cell3] If the histogram looks bimodal, set R_MIN at the trough manually, then run Cell 4.")

In [ ]:
# Cell 4 — Train v5.2 (verifiable-reward CoRPO) — fixes the v5.0/v5.1 pipeline defects
#
# Post-mortem of Runs #1-3 and v5.0/v5.1 found that NO earlier RL run tested the designed
# reward. v5.2 fixes, in order of impact:
#   1. max_prompt_length=6144 (corpo_train.py). TRL's GRPOConfig DEFAULTS to 512 and
#      left-truncates every prompt — ~87% of training prompts lost the system message
#      and most of the diff in every earlier run. The policy was scored on defects it
#      could not see.
#   2. --v4-merged: policy = merged-v4 weights + FRESH LoRA. TRL's KL reference for PEFT
#      models is "adapters disabled" — with the old adapter-loading that meant BASE
#      (pulling the policy away from v4); now it is actually v4.
#   3. Truncated rollouts (unclosed <think>) score 0 instead of leaking reasoning text
#      through the extractor into the reward.
#   4. Ambiguous "clean" records (no grounded defects but ungrounded defect comments)
#      are excluded — they were being trained as "find nothing" on diffs with real defects.
#   5. mid_eval now uses the same 12000-char budgeted prompts as the v4 baseline preds
#      and swaps checkpoints over merged-v4 (deltas are on merged-v4 now).
#   6. v5.2 reward constants: RECALL_BETA=1.5, CLEAN_CLAIM_PENALTY=0.35 (between v5.0's
#      over-quiet F1 and v5.1's over-loud beta=2).
#
# NOTE: checkpoints go to corpo-out-v5.2 — do NOT mix with corpo-out-v5: the old v5.0/v5.1
# checkpoints are LoRA deltas on base+v4-adapter, the new ones are deltas on merged-v4.

# Local, SHARD-VERIFIED copy of merged v4. A config.json-only check is not enough:
# Drive copies of 14 GB models get silently truncated (safetensors "header too small"
# at load — hit again 2026-06-10), so verify index total_size vs bytes on disk, and
# rebuild from the adapter (the ~300 MB source of truth) if both copies are bad.
import os, glob, json, shutil, gc

def _shards_ok(d):
    idx = os.path.join(d, 'model.safetensors.index.json')
    if not os.path.exists(idx):
        return False
    total = json.load(open(idx)).get('metadata', {}).get('total_size', 0)
    have = sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, '*.safetensors')))
    return total > 0 and have >= total

def _dir_chat_template(d):
    """Read a saved tokenizer's chat template straight from its files — the adapter's
    tokenizer files were written by a NEWER transformers and crash 4.56.2 on load."""
    p = os.path.join(d, 'chat_template.jinja')
    if os.path.exists(p):
        return open(p).read()
    return json.load(open(os.path.join(d, 'tokenizer_config.json'))).get('chat_template')

DRIVE_MERGED = '/content/drive/MyDrive/sft/sft-v4-merged-for-eval'
V4_MERGED    = '/content/sft-v4-merged'

if not _shards_ok(V4_MERGED):
    shutil.rmtree(V4_MERGED, ignore_errors=True)
    if _shards_ok(DRIVE_MERGED):
        print('[cell4] copying merged v4 from Drive (~3 min)...')
        !cp -r {DRIVE_MERGED} {V4_MERGED}
    if not _shards_ok(V4_MERGED):
        print('[cell4] Drive merged copy missing/corrupt — rebuilding from adapter (~10 min, CPU)...')
        shutil.rmtree(V4_MERGED, ignore_errors=True)
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel
        _b = AutoModelForCausalLM.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct', dtype=torch.bfloat16)
        _m = PeftModel.from_pretrained(_b, V4_ADAPTER).merge_and_unload()
        _m.save_pretrained(V4_MERGED, safe_serialization=True)  # local SSD = atomic
        # BASE tokenizer, not the adapter's: the adapter tokenizer files were written by
        # a newer transformers and 4.56.2 can't parse them. Safe because v4 never
        # customized the chat template — asserted right here.
        _tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')
        _tpl = _dir_chat_template(V4_ADAPTER)
        assert _tpl is None or _tpl == _tok.chat_template, 'adapter chat_template differs from base!'
        _tok.save_pretrained(V4_MERGED)
        del _m, _b, _tok; gc.collect()
        print('[cell4] rebuilt. AFTER training kicks off, refresh the Drive copy:')
        print(f'  !rm -rf {DRIVE_MERGED} && cp -r {V4_MERGED} {DRIVE_MERGED}')
assert _shards_ok(V4_MERGED), 'merged v4 failed shard-integrity check'
print(f"v4 merged (local, shard-verified): {V4_MERGED}")

!python /content/corpo_train.py \
    --v4-adapter {V4_ADAPTER} \
    --v4-merged {V4_MERGED} \
    --v4-backup {V4_BACKUP} \
    --train-prompts /content/ood_train_prompts.jsonl \
    --defect-labels /content/cache/defect_labels_train.jsonl \
    --output-dir /content/corpo-out \
    --checkpoint-sync-dir /content/drive/MyDrive/sft/corpo-out-v5.2 \
    --r-min-correct {R_MIN} \
    --kl-beta 0.02 \
    --learning-rate 5e-6 \
    --num-generations 8 \
    --prompts-per-step 4 \
    --max-new-tokens 2048 \
    --epochs 1 \
    --checkpoint-every 75 \
    --copy-to /content/drive/MyDrive/sft/code-reviewer-lora-v5.2-verifiable

# RESUME after a Colab disconnect: the local /content/corpo-out is wiped, but checkpoints
# are mirrored to Drive. Re-run Cells 0-1, re-run Cell 2 (labels) or restore the cache,
# then re-run THIS cell with --resume added, pointing at the latest Drive checkpoint, e.g.:
#   --resume /content/drive/MyDrive/sft/corpo-out-v5.2/checkpoint-150
# (batch note: per_device_train_batch_size=num_generations=8, gradient_accumulation_steps=
#  prompts_per_step=4 -> 4 prompts/optimizer-step; 8 % 8 == 0 satisfies TRL's divisibility rule.)

In [ ]:
# Cell 5 — mid-eval all v5.2 checkpoints vs the v4 bar (defect_recall / fp_rate / halluc)
#
# Runs as a SUBPROCESS: in-cell vLLM fails in Colab/Jupyter (vLLM v1 calls sys.stdout.fileno(),
# which a notebook stdout doesn't support). mid_eval.py LoRA-swaps each checkpoint over
# MERGED V4 (v5.2 checkpoints are deltas on merged-v4, NOT on the raw base), generates the
# fixed 50-prompt subset with the SAME 12000-char budgeted prompts the v4 preds used, and
# scores with the precision-aware judge-independent score_v5. Prints deltas vs the v4 bar.
!cd /content/sft && git pull --ff-only && cp /content/sft/*.py /content/
!python /content/mid_eval.py \
    --checkpoint-root /content/drive/MyDrive/sft/corpo-out-v5.2 \
    --v4-preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/cache/defect_labels_eval.jsonl \
    --v4-merged /content/sft-v4-merged \
    --n-samples 50
# Read the printed table: a checkpoint "beats v4" iff defect_recall > v4 AND fp_rate <= v4 AND halluc <= v4.
# n=50 is a DIRECTIONAL smoke test only (can't resolve <±0.10); the real call is Cell 7's
# full-632 paired bootstrap. Set BEST_CHECKPOINT to the winner (or /content/corpo-out/final
# after a full run) for Cell 6.

In [ ]:
# Cell 6 — Verify chat_template parity, merge BEST_CHECKPOINT onto MERGED V4, generate v5 preds
#
# Uses BEST_CHECKPOINT from Cell 5. Falls back to /content/corpo-out/final if Cell 5 was skipped.
# IMPORTANT (v5.2): checkpoints are LoRA deltas ON MERGED V4 — merging them onto the raw
# base would silently produce base+delta (the v4 weights would be missing).
import os, glob, json, shutil, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def _shards_ok(d):
    idx = os.path.join(d, 'model.safetensors.index.json')
    if not os.path.exists(idx):
        return False
    total = json.load(open(idx)).get('metadata', {}).get('total_size', 0)
    have = sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, '*.safetensors')))
    return total > 0 and have >= total

def _dir_chat_template(d):
    """Template straight from saved files — the adapter's tokenizer files were written
    by a NEWER transformers and crash 4.56.2 if loaded via AutoTokenizer."""
    p = os.path.join(d, 'chat_template.jinja')
    if os.path.exists(p):
        return open(p).read()
    return json.load(open(os.path.join(d, 'tokenizer_config.json'))).get('chat_template')

base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')

# 6a. chat_template parity (else run_ood_eval.py's assert fires mid-run)
_tpl = _dir_chat_template(V4_ADAPTER)
assert _tpl is None or _tpl == base_tok.chat_template, \
    "v4 chat_template differs from base — patch run_ood_eval.py to load tokenizer per-model"
print("[cell6] chat_template parity: OK")

# 6b. which checkpoint
try:
    _src = BEST_CHECKPOINT
    print(f"[cell6] using BEST_CHECKPOINT from Cell 5: {_src}")
except NameError:
    _src = '/content/corpo-out/final'
    print(f"[cell6] Cell 5 skipped — falling back to {_src}")

# 6c. ensure a SHARD-VERIFIED local merged v4 (config.json existing does not mean the
# 14 GB copy is complete)
DRIVE_MERGED = '/content/drive/MyDrive/sft/sft-v4-merged-for-eval'
V4_MERGED = '/content/sft-v4-merged'
if not _shards_ok(V4_MERGED):
    shutil.rmtree(V4_MERGED, ignore_errors=True)
    if _shards_ok(DRIVE_MERGED):
        !cp -r {DRIVE_MERGED} {V4_MERGED}
    if not _shards_ok(V4_MERGED):
        print('[cell6] rebuilding merged v4 from adapter...')
        shutil.rmtree(V4_MERGED, ignore_errors=True)
        _b = AutoModelForCausalLM.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct', dtype=torch.bfloat16)
        _m = PeftModel.from_pretrained(_b, V4_ADAPTER).merge_and_unload()
        _m.save_pretrained(V4_MERGED, safe_serialization=True)
        base_tok.save_pretrained(V4_MERGED)  # base tokenizer (parity asserted in 6a)
        del _m, _b; gc.collect()
assert _shards_ok(V4_MERGED), 'merged v4 failed shard-integrity check'

# merge the v5 checkpoint onto MERGED V4 (vLLM eval needs a full model)
base = AutoModelForCausalLM.from_pretrained(V4_MERGED, dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base, _src).merge_and_unload()
merged.save_pretrained('/content/sft-v5-merged-for-eval', safe_serialization=True)  # local first, Drive later
base_tok.save_pretrained('/content/sft-v5-merged-for-eval')
del merged, base
gc.collect(); torch.cuda.empty_cache()
assert _shards_ok('/content/sft-v5-merged-for-eval'), 'v5 merge failed shard-integrity check'

# 6d. generate v5 predictions on the 632 OOD set (--skip-base: base preds already in ood_preds_v4.jsonl)
!python /content/run_ood_eval.py \
    --input /content/ood_input.jsonl \
    --output /content/ood_preds_v5.jsonl \
    --v4-model /content/sft-v5-merged-for-eval \
    --skip-base

In [ ]:
# Cell 7 — v5 vs v4 on the FULL 632 OOD set (judge-independent) — PAIRED-significance verdict
#
# Two judge-independent views:
#   (A) POINT estimates (defect_recall / fp_rate / halluc) — directly comparable to the goal bar.
#   (B) PAIRED bootstrap (compare_recall.py) — per-record delta v5-v4 with a 95% CI. THE TRUSTWORTHY CALL.
#       Comparing two MEANS (view A) misses a real +0.04 recall gain ~90% of the time at this n
#       (per-record recall std ~0.4 over ~150 labeled records → SE on the mean ~0.033 > the gain).
#       The paired test cancels per-record difficulty and detects the same gain ~100% of the time.
#
# SHIP v5 iff: recall is a SIGNIFICANT paired improvement (CI lower bound > 0)
#             AND fp_rate not significantly worse AND halluc not significantly worse.
import json, sys
sys.path.insert(0, '/content')
import score_v5, compare_recall

labels = {}
for l in open('/content/cache/defect_labels_eval.jsonl'):
    if l.strip():
        r = json.loads(l); labels[r['instance_id']] = r.get('defects', [])

def _load(p):
    return [json.loads(l) for l in open(p) if l.strip()]

v4_preds = _load('/content/drive/MyDrive/sft/ood_preds_v4.jsonl')
v5_preds = _load('/content/ood_preds_v5.jsonl')   # run_ood_eval wrote v5 output under 'v4_pred'

# Backfill diff into v5 records from v4 (same instance_id) so grounding/halluc is computed
# on the REAL diff even if run_ood_eval didn't echo the diff field.
_diff_by = {r['instance_id']: r.get('diff', '') for r in v4_preds}
for r in v5_preds:
    if not r.get('diff'):
        r['diff'] = _diff_by.get(r['instance_id'], '')

# (A) POINT estimates — the literal-bar view
print("scoring v4 (632, parallel)..."); v4 = score_v5.score(v4_preds, labels, 'v4_pred')
print("scoring v5 (632, parallel)..."); v5 = score_v5.score(v5_preds, labels, 'v4_pred')
def _dr(s): return s['defect_recall_labeled'] if s['defect_recall_labeled'] is not None else 0.0
def _fp(s): return s['fp_rate_clean'] if s['fp_rate_clean'] is not None else 1.0
print(f"\n[POINT]  {'metric':16s} {'v4':>8} {'v5':>8} {'delta':>9}")
for name, fn in [('defect_recall', _dr), ('fp_rate(clean)', _fp), ('halluc', lambda s: s['halluc_mean'])]:
    a, b = fn(v4), fn(v5); print(f"         {name:16s} {a:>8.3f} {b:>8.3f} {b-a:>+9.3f}")

# (B) PAIRED bootstrap — the trustworthy verdict (both files store the model output under 'v4_pred')
print("\n[PAIRED] bootstrapping per-record deltas (v5 - v4), n_boot=2000 ...")
pd = compare_recall.paired_delta(v4_preds, v5_preds, labels, 'v4_pred', 'v4_pred', n_boot=2000, seed=0)
def _ci(c): return f"[{c[0]:+.4f}, {c[1]:+.4f}]"
print(f"         compared {pd['n_compared']} (labeled {pd['n_labeled']}, clean {pd['n_clean']})")
print(f"         recall  delta={pd['recall_delta']:+.4f}  CI={_ci(pd['recall_ci'])}  "
      f"{'SIG IMPROVEMENT' if pd['recall_significant'] else 'not sig'}")
print(f"         fp      delta={pd['fp_delta']:+.4f}  CI={_ci(pd['fp_ci'])}  "
      f"{'sig better' if pd['fp_significant'] else 'not sig'}")
print(f"         halluc  delta={pd['halluc_delta']:+.4f}  CI={_ci(pd['halluc_ci'])}  "
      f"{'sig better' if pd['halluc_significant'] else 'not sig'}")

# fp/halluc "not significantly worse" = their delta CI is NOT entirely above 0
fp_not_worse = pd['fp_ci'][0] <= 0
halluc_not_worse = pd['halluc_ci'][0] <= 0
ship = pd['recall_significant'] and fp_not_worse and halluc_not_worse
print(f"\nVERDICT: {'SHIP v5 — significant paired recall gain; fp & halluc not significantly worse' if ship else 'KEEP v4 — recall gain not significant (or fp/halluc significantly worse)'}")
print("(Paired bootstrap is the call; the POINT table above is the literal-bar view. n=632, judge-independent.)")
json.dump({'point': {'v4': v4, 'v5': v5}, 'paired': pd, 'ship': ship},
          open('/content/v5_final_verdict.json', 'w'), indent=2)
